In [11]:
import sys
import csv
import os
import json
import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# Load .env
load_dotenv(find_dotenv())

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = os.environ.get("DATA_DIR", "/workspace/data")

parsed_docs_dir = os.path.join(DATA_DIR, "parsed_docs")
train_csv_path = os.path.join(parsed_docs_dir, "training_final.csv")
test_csv_path = os.path.join(parsed_docs_dir, "test_final.csv")

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")
print(f"Train CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

SEED: 42
CUDA available: True
PyTorch version: 2.2.0
Data dir: /workspace/data
Train CSV path: /workspace/data/parsed_docs/training_final.csv
Test CSV path: /workspace/data/parsed_docs/test_final.csv


## Debugging

In [3]:
with open(train_csv_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if line.count('"') % 2 != 0:
            print(f"⚠️ Possible broken quotes at line {i}")
            print(line[:500])

⚠️ Possible broken quotes at line 379
"The secreted frizzled-related protein 3 (sFPR3) is altered in human cancers.

⚠️ Possible broken quotes at line 380
Are its level found to increase or to decrease?",summary,"BACKGROUND: Diagnosing obstructive sleep apnea (OSA) is clinically relevant because untreated OSA has been associated with increased morbidity and mortality. The STOP-Bang questionnaire is a validated screening tool for OSA. We conducted a systematic review and meta-analysis to determine the effectiveness of STOP-Bang for screening patients suspected of having OSA and to predict its accuracy in determining the severity of OSA in the diffe


KeyboardInterrupt: 

## Load the CSV and parse the abstracts column

In [20]:
# Increase CSV field limit
csv.field_size_limit(sys.maxsize)

# Read only first 10 rows
df_train_raw = pd.read_csv(train_csv_path, engine="python")
df_train = df_train_raw.copy()

In [21]:
print("Rows:", len(df_train))
print("Columns:", df_train.columns.tolist())

df_train.info()

Rows: 39418
Columns: ['question', 'type', 'text', 'category', 'topic_id', 'confidence', 'pmid', 'source_url', 'n_sources']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39418 entries, 0 to 39417
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   question    39418 non-null  object 
 1   type        39418 non-null  object 
 2   text        39417 non-null  object 
 3   category    39418 non-null  object 
 4   topic_id    39418 non-null  int64  
 5   confidence  39418 non-null  float64
 6   pmid        39418 non-null  int64  
 7   source_url  39418 non-null  object 
 8   n_sources   39418 non-null  int64  
dtypes: float64(1), int64(3), object(5)
memory usage: 2.7+ MB


In [26]:
df_train.isnull().sum()

question      0
type          0
text          1
category      0
topic_id      0
confidence    0
pmid          0
source_url    0
n_sources     0
dtype: int64

In [23]:
df_train["question"].value_counts()

question
Which is the causative agent of malaria?                                                                                                    105
In which proteins is the chromodomain present?                                                                                              102
What tyrosine kinase, involved in a Philadelphia- chromosome positive chronic myelogenous leukemia, is the target of Imatinib (Gleevec)?     79
Is miR-21 related to carcinogenesis?                                                                                                         69
What clinical conditions influence the prognostic after the liver metastasis resection from colorectal cancer patients?                      65
                                                                                                                                           ... 
What methodology does the HercepTest use?                                                                                      

In [27]:
df_train["category"].value_counts()

category
Other                    5690
Pharmacology & Drugs     5511
Molecular Biology        4503
Rare Diseases            3909
Cancer & Oncology        3587
Genetics & Mutations     3536
Infectious Disease       3145
Immunology               3036
Cardiology & Heart       2150
Surgery & Procedures     1632
Neurology & Brain        1345
Mental Health             445
Diagnostics & Imaging     393
Metabolism & Diabetes     353
Clinical Guidelines       183
Name: count, dtype: int64

In [28]:
import matplotlib.pyplot as plt

df["category"].value_counts().plot(kind="bar")
plt.title("Category Distribution")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [6]:
# Extract abstract dataframe
df_abstracts = df[["pmid", "text"]].copy()

# Drop duplicate PMIDs
df_abstracts = df_abstracts.drop_duplicates(subset=["pmid"]).reset_index(drop=True)

print("Rows after dedup:", len(df_abstracts))
print()

# Display nicely
for i, row in df_abstracts.head(5).iterrows():
    print("="*80)
    print(f"PMID: {row['pmid']}")
    print("-"*80)
    print(row["text"])
    print(len(row["text"]))
    print()

Rows after dedup: 95

PMID: 15829955
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    OBJECTIVE: To analyze the prognostic factors o...
Name: 0, dtype: object
2

PMID: 15617541
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Neuromyelitis Optica Spectrum Diso...
Name: 1, dtype: object
2

PMID: 12239580
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Severe forms of primary dystonia a...
Name: 2, dtype: object
2

PMID: 6650562
--------------------------------------------------------------------------------
text    OBJECTIVE: To analyze the prognostic factors o...
text    BACKGROUND: Recent studies linking radiation e...
Name: 3, dtype: object
2

PMID: 20598273
------------

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.embedding_manager import MedicalEmbeddingManager

embedding_manager = MedicalEmbeddingManager()

query_vec = embedding_manager.embed_query(
    "What causes Alzheimer's disease?"
)

chunks = [
    "Amyloid beta accumulation is a key factor in Alzheimer's disease.",
    "Neuroinflammation contributes to neurodegenerative disorders.",
    "Tau protein aggregation leads to neuronal damage."
]

doc_vecs = embedding_manager.embed_documents(chunks)

print("Query embedding dimension:", len(query_vec))
print("First 10 values of query embedding:", query_vec[:10])

print("\nNumber of document embeddings:", len(doc_vecs))
print("Document embedding dimension:", len(doc_vecs[0]))
print("First 10 values of first document embedding:", doc_vecs[0][:10])

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Query embedding dimension: 768
First 10 values of query embedding: [-0.02226661890745163, -0.04278154671192169, -0.0294787660241127, -0.02906412072479725, 0.004392766393721104, 0.004281898029148579, -0.01897631213068962, 0.026256967335939407, 0.014577195979654789, 0.0341288186609745]

Number of document embeddings: 3
Document embedding dimension: 768
First 10 values of first document embedding: [-0.031111201271414757, -0.03551964461803436, -0.03676166757941246, -0.04713869467377663, -0.008105887100100517, -0.00807227473706007, -0.030369000509381294, 0.03008858673274517, 0.017857875674962997, 0.027921907603740692]


In [9]:
import torch
import transformers
import sentence_transformers

print(torch.__version__)
print(transformers.__version__)
print(sentence_transformers.__version__)
print(embedding_manager.embedding_model)

2.2.0
4.41.2
2.7.0
pritamdeka/S-PubMedBert-MS-MARCO
